### 1. Environment Setup
Start your Jupyter notebook and install the necessary Python packages. The most important libraries are transformers, datasets, peft, trl, and optionally tracking tools like Weights & Biases.

In [ ]:
!pip install -U transformers datasets accelerate peft trl bitsandbytes wandb


In [ ]:
import pandas as pd
from datasets import Dataset

df = pd.read_csv("instruction-response.csv")  # Ensure columns: instruction, response
dataset = Dataset.from_pandas(df)





In [ ]:
from huggingface_hub import login
login(token="YOUR_HUGGINGFACE_TOKEN")

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, load_in_8bit=True, device_map="auto")


In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)


### 6. Training Arguments
Set up the training configuration:

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./llama-chatbot-finetuned",
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=1,
    learning_rate=2e-4,
    fp16=True,
    evaluation_strategy="steps",
    save_strategy="steps",
    save_steps=100,
    logging_steps=10,
)


### 7. Supervised Fine-Tuning Trainer Setup
Use the SFTTrainer or Trainer class for supervised fine-tuning:

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["validation"],
    tokenizer=tokenizer,
    args=training_args,
    packing=False,
    max_seq_length=512,
    dataset_text_field="instruction"
)


### 8. Launch Training
Start the fine-tuning process:

In [ ]:
trainer.train()

### 9. Saving and Exporting the Fine-Tuned Model
After training, save your model for later use in a chatbot application:



In [ ]:
trainer.save_model("./llama-chatbot-finetuned")
tokenizer.save_pretrained("./llama-chatbot-finetuned")
